Tên chiến lược: Dynamic Trend-Following ATR

Vị thế (Position): LONG

Điều kiện vào (Entry):

MA100 > MA200: Xác nhận thị trường vĩ mô đang trong Uptrend.

Giá Đóng (Close) cắt LÊN MA50: Xác nhận nhịp điều chỉnh (Pullback) đã kết thúc, giá bắt đầu tăng trở lại theo xu hướng chính.

Dừng lỗ (Stop Loss):

SL = Giá Entry - (2 × ATR) (Tránh bị quét râu nến do nhiễu thị trường).

Chốt lời (Take Profit):

TP = Giá Entry + (4 × ATR) (Đảm bảo tỷ lệ R:R chuẩn 1:2).

# Python Code

In [ ]:
!pip install pandas pandas-ta requests plotly -q

import requests
import time
import pandas as pd
import pandas_ta as ta
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime


# ==========================================
# GLOBAL CONFIGURATION
# ==========================================
CONFIG = {
    'SYMBOLS': ['BTCUSDT', 'BNBUSDT'],
    'INTERVAL': '1h',
    'START_DATE': '01/04/2020',
    'END_DATE': '01/04/2021',
    'INITIAL_CAPITAL': 10000.0,
    'POSITION_SIZE_PCT': 0.10,
    'FEE_RATE': 0.0004,
    'SLIPPAGE': 0.0002
}

# ==========================================
# MODULE 1: DATA HANDLER
# ==========================================
class BinanceDataFetcher:
    def __init__(self, symbol, interval, start_date, end_date):
        self.symbol = symbol.upper()
        self.interval = interval
        self.start_date = start_date
        self.end_date = end_date

    def _str_to_millis(self, date_str: str) -> int:
        dt = datetime.strptime(date_str, "%d/%m/%Y")
        return int(dt.timestamp() * 1000)

    def fetch(self) -> pd.DataFrame:
        base_url = "https://data-api.binance.vision"
        req_symbol = self.symbol

        print(f"Checking connection for {req_symbol}...")
        try:
            test = requests.get(f"{base_url}/api/v3/time", timeout=5)
            if test.status_code == 451:
                print("US IP detected. Switching to Binance.US...")
                base_url = "https://api.binance.us"
                if req_symbol == "ETHUSDT": req_symbol = "ETHUSD"
                if req_symbol == "BNBUSDT": req_symbol = "BNBUSD"
        except Exception:
            base_url = "https://api.binance.us"
            if req_symbol == "ETHUSDT": req_symbol = "ETHUSD"
            if req_symbol == "BNBUSDT": req_symbol = "BNBUSD"

        url = base_url + "/api/v3/klines"
        start_time = self._str_to_millis(self.start_date)

        if self.end_date == datetime.now().strftime("%d/%m/%Y"):
            end_time = int(time.time() * 1000)
        else:
            end_dt = datetime.strptime(self.end_date, "%d/%m/%Y").replace(hour=23, minute=59, second=59)
            end_time = int(end_dt.timestamp() * 1000)

        all_klines =[]
        print(f"Downloading {req_symbol} ({self.interval})...")

        while start_time < end_time:
            params = {"symbol": req_symbol, "interval": self.interval, "startTime": start_time, "endTime": end_time, "limit": 1000}
            try:
                r = requests.get(url, params=params, timeout=10)
                if r.status_code != 200: break
                data = r.json()
                if not data: break
                all_klines.extend(data)
                start_time = data[-1][0] + 1
                time.sleep(0.05)
            except Exception: break

        if not all_klines: return pd.DataFrame()

        df = pd.DataFrame(all_klines, columns=["timestamp", "Open", "High", "Low", "Close", "Volume", "CT", "QAV", "NT", "TBBV", "TBQV", "I"])
        df = df[["timestamp", "Open", "High", "Low", "Close", "Volume"]]
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
        df.set_index("timestamp", inplace=True)
        df = df.astype(float)
        print(f"Total: {len(df)} candles fetched.")
        return df

# ==========================================
# MODULE 2: STRATEGY LOGIC
# ==========================================
class DynamicTrendATRStrategy:
    def __init__(self):
        self.ma_fast = 50
        self.ma_mid = 100
        self.ma_slow = 200
        self.atr_len = 14

        # Hệ số TP / SL
        self.sl_multiplier = 2.0  # Dừng lỗ = 2 * ATR
        self.tp_multiplier = 4.0  # Chốt lời = 4 * ATR (Tỷ lệ R:R = 1:2)

    def calculate_indicators(self, df: pd.DataFrame) -> pd.DataFrame:
        # Tính toán Moving Averages (Dùng SMA cho hệ thống Trend Following cơ bản)
        df['MA50'] = ta.sma(df['Close'], length=self.ma_fast)
        df['MA100'] = ta.sma(df['Close'], length=self.ma_mid)
        df['MA200'] = ta.sma(df['Close'], length=self.ma_slow)

        # Tính toán ATR
        df['ATR'] = ta.atr(df['High'], df['Low'], df['Close'], length=self.atr_len)

        # Bỏ các nến ban đầu bị NaN
        df.dropna(inplace=True)
        return df

# ==========================================
# MODULE 3: BACKTEST ENGINE (DYNAMIC SL/TP)
# ==========================================
class BacktestEngine:
    def __init__(self, df, strategy, config):
        self.df = strategy.calculate_indicators(df.copy())
        self.strategy = strategy
        self.config = config

        self.capital = config['INITIAL_CAPITAL']
        self.invested_amount = 0.0
        self.position = 0
        self.entry_price = 0.0

        # Lưu trữ mức giá SL và TP
        self.sl_price = 0.0
        self.tp_price = 0.0
        self.trades = []

        # Khởi tạo toàn bộ mảng bằng số vốn ban đầu (Tránh lỗi NaN Sharpe Ratio)
        self.equity_curve = np.full(len(self.df), self.capital)

    def run(self):
        O = self.df['Open'].values
        H = self.df['High'].values
        L = self.df['Low'].values
        C = self.df['Close'].values

        ma50 = self.df['MA50'].values
        ma100 = self.df['MA100'].values
        ma200 = self.df['MA200'].values
        atr = self.df['ATR'].values
        times = self.df.index

        fee = self.config['FEE_RATE']
        slip = self.config['SLIPPAGE']
        pos_size_pct = self.config['POSITION_SIZE_PCT']

        s = self.strategy

        for i in range(2, len(self.df)):

            # --- 1. EXIT LOGIC ---
            if self.position == 1:
                exit_signal = False
                reason = ""
                exit_price = 0.0

                # KIỂM TRA STOP LOSS TRƯỚC (Bảo thủ - Tránh Wick hai chiều)
                if L[i] <= self.sl_price:
                    exit_signal = True
                    reason = "Stop Loss (-2 ATR)"
                    # Nếu Gap Down thủng SL, khớp ở Open. Ngược lại khớp tại SL.
                    exit_price = min(self.sl_price, O[i])

                # KIỂM TRA TAKE PROFIT
                elif H[i] >= self.tp_price:
                    exit_signal = True
                    reason = "Take Profit (+4 ATR)"
                    # Nếu Gap Up vượt TP, khớp ở Open. Ngược lại khớp tại TP.
                    exit_price = max(self.tp_price, O[i])

                if exit_signal:
                    real_exit = exit_price * (1 - slip)
                    pnl_pct = (real_exit - self.entry_price) / self.entry_price

                    net_pnl_pct = pnl_pct - (fee * 2)
                    pnl_amt = self.invested_amount * net_pnl_pct
                    self.capital += pnl_amt

                    self.trades.append({
                        'Date': times[i], 'Type': 'Long',
                        'Reason': reason,
                        'Entry': self.entry_price, 'Exit': real_exit,
                        'PnL_Amt': pnl_amt, 'Equity': self.capital
                    })

                    self.position = 0
                    self.invested_amount = 0.0
                    self.sl_price = 0.0
                    self.tp_price = 0.0

            # --- 2. ENTRY LOGIC ---
            elif self.position == 0:

                # Điều kiện 1: Uptrend dài hạn
                cond_uptrend = ma100[i-1] > ma200[i-1]

                # Điều kiện 2: Giá Đóng (Close) cắt LÊN MA50
                cond_cross = (C[i-1] > ma50[i-1]) and (C[i-2] <= ma50[i-2])

                if cond_uptrend and cond_cross:
                    self.position = 1
                    self.entry_price = O[i] * (1 + slip)
                    self.invested_amount = self.capital * pos_size_pct

                    # Tính toán Stop Loss và Take profit dựa trên nến tín hiệu [i-1]
                    current_atr = atr[i-1]
                    self.sl_price = self.entry_price - (s.sl_multiplier * current_atr)
                    self.tp_price = self.entry_price + (s.tp_multiplier * current_atr)

            # --- 3. TÍNH UNREALIZED PNL ---
            current_equity = self.capital
            if self.position == 1:
                unrealized_pnl_pct = (C[i] - self.entry_price) / self.entry_price
                unrealized_pnl = self.invested_amount * unrealized_pnl_pct
                current_equity += unrealized_pnl

            self.equity_curve[i] = current_equity

        return pd.Series(self.equity_curve, index=self.df.index), pd.DataFrame(self.trades)

# ==========================================
# MODULE 4: PERFORMANCE EVALUATOR
# ==========================================
class Evaluator:
    @staticmethod
    def generate_report(equity, trades_df, initial_capital, symbol, interval, pos_size_pct):
        if trades_df.empty:
            print(f"No trades executed for {symbol}!")
            return

        final_equity = equity.iloc[-1]
        net_pnl = final_equity - initial_capital
        total_return = net_pnl / initial_capital

        rolling_max = equity.cummax()
        drawdown = (equity - rolling_max) / rolling_max
        max_dd = drawdown.min()

        wins = trades_df[trades_df['PnL_Amt'] > 0]
        losses = trades_df[trades_df['PnL_Amt'] <= 0]
        winrate = len(wins) / len(trades_df) if len(trades_df) > 0 else 0
        profit_factor = wins['PnL_Amt'].sum() / abs(losses['PnL_Amt'].sum()) if len(losses) > 0 and losses['PnL_Amt'].sum() != 0 else 0

        days = (equity.index[-1] - equity.index[0]).days
        cagr = ((final_equity / initial_capital) ** (365.0 / days)) - 1 if days > 0 else 0

        returns = equity.pct_change().dropna()
        sharpe_ratio = (returns.mean() / returns.std()) * np.sqrt(365 * 24) if returns.std() != 0 else 0

        print("")
        print(f"========== PERFORMANCE REPORT: {symbol} ({interval}) ==========")
        print(f"{'Strategy:':>20} Dynamic Trend-Following ATR")
        print(f"{'Position Sizing:':>20} {pos_size_pct*100}% of Capital")
        print(f"{'Initial Capital:':>20} ${initial_capital:,.2f}")
        print(f"{'Final Equity:':>20} ${final_equity:,.2f}")
        print(f"{'Net PnL:':>20} ${net_pnl:,.2f}")
        print(f"{'Total Return:':>20} {total_return:.2%}")
        print(f"{'CAGR (Annual):':>20} {cagr:.2%}")
        print(f"{'Max Drawdown:':>20} {max_dd:.2%}")
        print(f"{'Sharpe Ratio:':>20} {sharpe_ratio:.2f}")
        print(f"{'Total Trades:':>20} {len(trades_df)}")
        print(f"{'Winrate:':>20} {winrate:.2%}")
        print(f"{'Profit Factor:':>20} {profit_factor:.2f}")
        print("====================================================\n")

        fig = make_subplots(
            rows=2, cols=1,
            shared_xaxes=True,
            vertical_spacing=0.08,
            row_heights=[0.7, 0.3]
        )

        fig.add_trace(go.Scatter(
            x=equity.index, y=equity.values,
            mode="lines", name="Equity",
            line=dict(color="#2962ff", width=1.5)
        ), row=1, col=1)

        fig.add_trace(go.Scatter(
            x=drawdown.index, y=drawdown.values,
            mode="lines", name="Drawdown",
            line=dict(color="#d32f2f", width=1),
            fill='tozeroy', fillcolor='rgba(211, 47, 47, 0.2)'
        ), row=2, col=1)

        fig.update_layout(
            title=f"Portfolio Performance: {symbol} ({interval})",
            template="plotly_white",
            height=650,
            margin=dict(l=40, r=40, t=60, b=40),
            showlegend=False
        )

        fig.update_xaxes(showgrid=True, gridcolor='#e6e6e6', zeroline=False)
        fig.update_yaxes(title_text="Balance ($)", tickprefix="$", showgrid=True, gridcolor='#e6e6e6', zeroline=False, row=1, col=1)
        fig.update_yaxes(title_text="Drawdown %", tickformat=".1%", showgrid=True, gridcolor='#e6e6e6', zeroline=True, zerolinecolor='#cccccc', row=2, col=1)

        fig.show()

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    for symbol in CONFIG['SYMBOLS']:
        print(f"\n---> Starting Backtest for: {symbol} <---")
        fetcher = BinanceDataFetcher(symbol, CONFIG['INTERVAL'], CONFIG['START_DATE'], CONFIG['END_DATE'])
        df_raw = fetcher.fetch()

        if not df_raw.empty:
            # Khởi tạo chiến lược mới
            my_strategy = DynamicTrendATRStrategy()

            engine = BacktestEngine(df_raw, my_strategy, CONFIG)
            equity, trades = engine.run()

            Evaluator.generate_report(
                equity,
                trades,
                CONFIG['INITIAL_CAPITAL'],
                symbol,
                CONFIG['INTERVAL'],
                CONFIG['POSITION_SIZE_PCT']
            )


---> Starting Backtest for: BTCUSDT <---
Checking connection for BTCUSDT...
Total: 8771 candles fetched.

========== PERFORMANCE REPORT: BTCUSDT (1h) ==========
           Strategy: Dynamic Trend-Following ATR
    Position Sizing: 10.0% of Capital
    Initial Capital: $10,000.00
       Final Equity: $10,624.45
            Net PnL: $624.45
       Total Return: 6.24%
      CAGR (Annual): 6.39%
       Max Drawdown: -1.81%
       Sharpe Ratio: 1.87
       Total Trades: 97
            Winrate: 44.33%
      Profit Factor: 1.47




---> Starting Backtest for: BNBUSDT <---
Checking connection for BNBUSDT...
Total: 8771 candles fetched.

========== PERFORMANCE REPORT: BNBUSDT (1h) ==========
           Strategy: Dynamic Trend-Following ATR
    Position Sizing: 10.0% of Capital
    Initial Capital: $10,000.00
       Final Equity: $10,467.73
            Net PnL: $467.73
       Total Return: 4.68%
      CAGR (Annual): 4.78%
       Max Drawdown: -3.77%
       Sharpe Ratio: 1.02
       Total Trades: 83
            Winrate: 40.96%
      Profit Factor: 1.30



# Go Code